# Étape 2 : Construction du graphe biparti CV–Job

Ce notebook construit uniquement le graphe biparti initial à partir des correspondances
observées. 

## 1. Chargement des données

In [1]:
from pathlib import Path
from pprint import pprint
import json
import pickle

import networkx as nx
import pandas as pd
from datasets import load_dataset

DATASET_ID = "michaelozon/candidate-matching-synthetic"
ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
PROCESSED_DIR = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def load_table(name):
    return load_dataset(DATASET_ID, data_dir=name, split="train").to_pandas()

df_resumes = load_table("resumes")
df_jobs = load_table("jobs")
df_matches = load_table("matches")

dimensions = {
    "resumes": df_resumes.shape,
    "jobs": df_jobs.shape,
    "matches": df_matches.shape,
}
display(pd.Series(dimensions, name="dimensions").to_frame())

assert len(df_resumes) == 10_000
assert len(df_jobs) == 2_500
assert len(df_matches) == 2_500

,dimensions
resumes,"(10000, 9)"
jobs,"(2500, 9)"
matches,"(2500, 2)"


## 2. Couples CV–Job au format long

`explode()` transforme chaque liste de CV pertinents en lignes individuelles. Chaque ligne de
`match_pairs` correspondra directement à une arête.

In [2]:
match_pairs = (
    df_matches[["job_id", "relevant_resume_ids"]]
    .explode("relevant_resume_ids", ignore_index=True)
    .rename(columns={"relevant_resume_ids": "resume_id"})
    [["job_id", "resume_id"]]
)

display(match_pairs.head())
print("Dimensions de match_pairs :", match_pairs.shape)
assert len(match_pairs) == 75_000
assert not match_pairs.duplicated(["job_id", "resume_id"]).any()

,job_id,resume_id
0,J_000000,R_000915
1,J_000000,R_003468
2,J_000000,R_003641
3,J_000000,R_003326
4,J_000000,R_008830


Dimensions de match_pairs : (75000, 2)


## 3. Création du graphe et ajout des nœuds

In [3]:
# Les identifiants des deux partitions doivent être disjoints.
cv_ids = set(df_resumes["resume_id"])
job_ids = set(df_jobs["job_id"])
assert cv_ids.isdisjoint(job_ids)

G = nx.Graph()

def python_value(value):
    # Conserve les attributs et convertit les tableaux Hugging Face en listes Python.
    if hasattr(value, "tolist"):
        return value.tolist()
    if hasattr(value, "item"):
        return value.item()
    return value

for record in df_resumes.to_dict(orient="records"):
    resume_id = record.pop("resume_id")
    attributes = {key: python_value(value) for key, value in record.items()}
    attributes.update(node_type="CV", bipartite=0)
    G.add_node(resume_id, **attributes)

for record in df_jobs.to_dict(orient="records"):
    job_id = record.pop("job_id")
    attributes = {key: python_value(value) for key, value in record.items()}
    attributes.update(node_type="Job", bipartite=1)
    G.add_node(job_id, **attributes)

print("Nœuds ajoutés :", G.number_of_nodes())

Nœuds ajoutés : 12500


## 4. Ajout des arêtes observées

In [4]:
G.add_edges_from(match_pairs[["resume_id", "job_id"]].itertuples(index=False, name=None))
print("Arêtes ajoutées :", G.number_of_edges())

Arêtes ajoutées : 75000


## 5. Vérifications 

Ces contrôles valident uniquement la construction et le caractère biparti du graphe.

In [5]:
n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()
n_cv_nodes = sum(data["node_type"] == "CV" for _, data in G.nodes(data=True))
n_job_nodes = sum(data["node_type"] == "Job" for _, data in G.nodes(data=True))
is_bipartite = nx.is_bipartite(G)
n_self_loops = nx.number_of_selfloops(G)
n_cv_cv_edges = sum(
    G.nodes[u]["node_type"] == "CV" and G.nodes[v]["node_type"] == "CV"
    for u, v in G.edges()
)
n_job_job_edges = sum(
    G.nodes[u]["node_type"] == "Job" and G.nodes[v]["node_type"] == "Job"
    for u, v in G.edges()
)

graph_summary = {
    "n_nodes": n_nodes,
    "n_edges": n_edges,
    "n_cv_nodes": n_cv_nodes,
    "n_job_nodes": n_job_nodes,
    "is_bipartite": bool(is_bipartite),
    "n_self_loops": n_self_loops,
    "n_cv_cv_edges": n_cv_cv_edges,
    "n_job_job_edges": n_job_job_edges,
}
display(pd.Series(graph_summary, name="valeur").to_frame())

assert n_nodes == 12_500
assert n_edges == 75_000
assert n_cv_nodes == 10_000
assert n_job_nodes == 2_500
assert is_bipartite
assert n_self_loops == 0
assert n_cv_cv_edges == 0
assert n_job_job_edges == 0
print("Toutes les assertions sont passées.")

,valeur
n_nodes,12500
n_edges,75000
n_cv_nodes,10000
n_job_nodes,2500
is_bipartite,True
n_self_loops,0
n_cv_cv_edges,0
n_job_job_edges,0


Toutes les assertions sont passées.


## 6. Vérification d’un CV et d’un Job reliés

In [6]:
example_resume_id, example_job_id = next(iter(G.edges()))
if G.nodes[example_resume_id]["node_type"] != "CV":
    example_resume_id, example_job_id = example_job_id, example_resume_id

print("CV :", example_resume_id)
pprint(G.nodes[example_resume_id], sort_dicts=False)
print("\nJob :", example_job_id)
pprint(G.nodes[example_job_id], sort_dicts=False)
print("\nArête présente :", G.has_edge(example_resume_id, example_job_id))
assert G.has_edge(example_resume_id, example_job_id)

CV : R_000000
{'role': 'Software Engineer',
 'seniority': 'Senior',
 'years_experience': 12,
 'industry': 'EdTech',
 'education': 'BSc',
 'skills': ['OOP',
            'Databases',
            'Git',
            'Docker',
            'Python',
            'Unit Testing',
            'Java'],
 'summary': 'Software Engineer with 12 years of experience in EdTech.',
 'experience_bullets': ['Delivered results using structured workflows and '
                        'clear communication',
                        'Collaborated with stakeholders to define needs and '
                        'execute tasks',
                        'Maintained reporting and documentation to support '
                        'team performance'],
 'node_type': 'CV',
 'bipartite': 0}

Job : J_000324
{'job_title': 'Full Stack Engineer',
 'seniority': 'Junior',
 'industry': 'Travel',
 'must_have_skills': ['JavaScript', 'OOP', 'Databases'],
 'nice_to_have_skills': ['Git'],
 'description': 'We are hiring a Full Stack 

## 7. Sauvegarde du graphe et du résumé

In [7]:
graph_path = PROCESSED_DIR / "cv_job_graph.pkl"
summary_path = RESULTS_DIR / "step2_graph_summary.json"

with graph_path.open("wb") as file:
    pickle.dump(G, file, protocol=pickle.HIGHEST_PROTOCOL)

summary_path.write_text(
    json.dumps(graph_summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Graphe sauvegardé :", graph_path)
print("Résumé sauvegardé :", summary_path)

Graphe sauvegardé : data\processed\cv_job_graph.pkl
Résumé sauvegardé : results\step2_graph_summary.json
